# Phase 2 — Sequence Learning and Comparison

This notebook documents the completed FD001 LSTM experiment and reads its saved artifacts. It does **not** retrain the model or rerun the official test evaluation: the 100-engine official test set was evaluated once after validation-based model selection.

The sequence pipeline splits by engine ID before creating windows, fits sensor normalization on training engines only, preserves chronological order and engine boundaries, left-pads short histories by repeating the first normalized sensor vector, and aligns each label to the window's final timestep. Training targets are capped at 125 cycles. Test evaluation uses one final observed window per official test engine and the uncapped supplied RUL.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Run this notebook from the repository root or notebooks/.")

RESULTS = ROOT / "outputs" / "results" / "phase2"
FIGURES = ROOT / "outputs" / "figures" / "phase2"
print("Repository root:", ROOT.name)
print("Phase 2 result files:", len(list(RESULTS.glob("*"))))

## Dataset and engine-level split

Only the run-to-failure training file is loaded in the validation walkthrough. The official test file is intentionally not read in this notebook. The saved selection record captures the seeded 80/20 split of training engine IDs and the validation cutoff cycles.

In [ ]:
from src.data import add_train_rul, load_train_cmapss
from src.sequence_data import EngineStandardizer, make_sequence_windows, split_engine_ids

train = add_train_rul(load_train_cmapss("FD001"))
selection = json.loads((RESULTS / "selection.json").read_text())
train_ids, validation_ids = split_engine_ids(train["unit"].unique(), validation_fraction=0.2, seed=42)
assert set(train_ids).isdisjoint(validation_ids)
assert set(train_ids) | set(validation_ids) == set(train["unit"].unique())
assert train_ids.tolist() == selection["training_engine_ids"]
assert validation_ids.tolist() == selection["validation_engine_ids"]
print(f"Training rows: {len(train):,}; engines: {train['unit'].nunique()}")
print(f"Fit engines: {len(train_ids)}; held-out validation engines: {len(validation_ids)}")
print("Validation cutoff cycle range:", min(selection["validation_cutoff_cycles"].values()), "to", max(selection["validation_cutoff_cycles"].values()))

## Sequence construction and preprocessing

The following small batch demonstrates tensor and target alignment without creating a full dataset of overlapping windows. The scaler is fitted over the 80 fit engines and then applied to three fit-engine histories. The experiment runner similarly creates windows only after selecting disjoint engine groups.

In [ ]:
selected = selection["selected_experiment"]
window_size = selected["window_size"]
standardizer = EngineStandardizer().fit(train, train_ids)
sample_ids = train_ids[:3]
sample_rows = train.loc[train["unit"].isin(sample_ids)]
sample_normalized = standardizer.transform(sample_rows)
sample_windows = make_sequence_windows(
    sample_normalized,
    window_size=window_size,
    target_cap=125,
)
print("Window tensor shape:", sample_windows.x.shape)
print("Target shape:", sample_windows.y.shape)
print("First endpoint (engine, cycle, capped RUL):", sample_windows.unit[0], sample_windows.cycle[0], sample_windows.y[0])
assert sample_windows.x.shape[1:] == (window_size, len(standardizer.sensor_columns))

## Selected LSTM architecture

A single recurrent layer encodes the final 50-cycle, 14-sensor window. Its last hidden representation passes through a dropout/fully connected regression head to produce one continuous RUL estimate. Selection used capped-RUL validation RMSE on the 20 held-out training engines, not official test performance.

In [ ]:
import torch
from src.lstm import LSTMRegressor

model = LSTMRegressor(**selected["model_config"])
with torch.no_grad():
    example_output = model(torch.from_numpy(sample_windows.x[:2]))
print(model)
print("Example output shape:", tuple(example_output.shape))
print("Selected validation RMSE / MAE:", selected["validation_rmse_capped"], selected["validation_mae_capped"])

## Validation experiments

The sweep varied window length (20, 30, 50), dropout, and weight decay. The best validation configuration was window length 50 with dropout 0.2 and weight decay 0.0001. These runs are model-selection evidence only; they are not test results.

In [ ]:
validation = pd.read_csv(RESULTS / "validation_experiments.csv")
display(validation[[
    "name", "window_size", "dropout", "weight_decay",
    "validation_rmse_capped", "validation_mae_capped", "best_epoch"
]].sort_values("validation_rmse_capped").reset_index(drop=True))

## Frozen official-test evaluation and Phase 1 comparison

The saved official-test results use one prediction per engine at its last observed cycle. Both models are compared on the same 100 test engines and the same uncapped true RUL values. Training labels are capped at 125; therefore, early-life test errors against true RUL above 125 include the intended target-cap mismatch. The capped-truth metrics are also recorded separately in `test_metrics.json`.

In [ ]:
test_metrics = json.loads((RESULTS / "test_metrics.json").read_text())
comparison = pd.read_csv(RESULTS / "model_comparison.csv")
display(comparison[["model", "variant", "RMSE", "MAE", "NASA_score", "RMSE 95% CI"]])
print("LSTM true-RUL metrics:", test_metrics["metrics_true_RUL"])
print("LSTM capped-RUL metrics:", test_metrics["metrics_capped_RUL"])
print("Selection protocol:", test_metrics["selection_protocol"])

In [ ]:
from IPython.display import display, Image

for name in [
    "final_training_loss.png",
    "model_comparison.png",
    "test_predictions.png",
    "test_residuals.png",
    "test_engine_errors.png",
    "test_stage_errors.png",
]:
    path = FIGURES / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))

In [ ]:
stage_errors = pd.read_csv(RESULTS / "test_error_by_stage.csv")
display(stage_errors)
lstm_rmse = test_metrics["metrics_true_RUL"]["RMSE"]
xgb_rmse = test_metrics["phase1_xgboost_same_test_population"]["RMSE"]
print(f"Official test RMSE difference: LSTM - XGBoost = {lstm_rmse - xgb_rmse:.2f} cycles ({(lstm_rmse / xgb_rmse - 1) * 100:.1f}% higher for LSTM).")
print("Conclusion: this validation-selected LSTM did not improve on the Phase 1 XGBoost benchmark. No post-test tuning was performed.")